# Analyse de donnees

- **Objectif :** explorer le jeu de données afin de valider la pertinence des variables statiques et démographiques avant la phase de modélisation prédictive.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from config import FEATURES
from tableone import TableOne
from scipy.stats import mannwhitneyu, chi2_contingency, false_discovery_control

## Preparation des donnees

1. Chargement des donnees

In [ ]:
data = pd.read_csv(FEATURES)
data.head()

2. Exclusion des variables redendantes

In [ ]:
redundant_cols = ['id_programme','resultat_final','date_debut','date_annulation', ]
missing_cols = ['avg_note','avg_lateness_days']
data.drop(redundant_cols+missing_cols, axis='columns', inplace=True)
data.head()

3. Type des variables

In [ ]:
num = ['nb_inscriptions_precedentes','age','total_volume','distinct_active_days','days_since_last_activity','n_evaluations']
cat = ['genre','region','situation_handicap','niveau_scolaire','annee_scolaire','cohorte']
churn = 'churn'

## Visualisation des donnees 

1. Les variables numeriques

In [ ]:
data[num].describe()

In [ ]:
# Boxplot
for col in num:
    sns.boxplot(data=data, x=churn, y=col)
    plt.show()

2. Les variables Categoriques

In [ ]:
print(data[churn].value_counts(normalize=True))
for col in cat:
    print(f'Pour la variable {col}:')
    print(data[col].value_counts())
    print(pd.crosstab(data[col],data['churn'],normalize='index'))
    print('\n')


In [ ]:
# visualisation
base_rate = data[churn].mean()
for col in cat:
    churn_rate = data.groupby(col)[churn].mean().sort_values(ascending=False)

    sns.barplot(x=churn_rate.values, y=churn_rate.index, orient='h', color='steelblue')
    plt.axvline(base_rate, color='red', linestyle='--', label=f'Taux d\'abandonnement global ({base_rate:.1%})')
    plt.xlabel('Taux d\'abandonnement')
    plt.ylabel(col)
    plt.title(f'Taux d\'abandonnement par categorie de {col}')
    plt.legend()
    plt.tight_layout()
    plt.show()

## Significativite des variables

In [ ]:
cols = num + cat 
table = TableOne(
    data,
    columns=cols,
    nonnormal=num,
    categorical=cat,
    groupby=churn,
    pval=True,
    htest_name=True
)
table

1. Test de Mann-Whitney pour les variables numeriques

> **$H_0$:** la distribution de la variable est identique entre les etudiants qui abandonnent (churn=1) et ceux qui n'abandonnent pas (churn=0).

> **$H_1$:** les deux distributions different (l'un des deux groupes tend a avoir des valeurs plus elevees que l'autre).

In [ ]:
res = []
for col in num:
    group0=data.loc[data[churn]==0,col]
    group1=data.loc[data[churn]==1,col]
    n0,n1=len(group0), len(group1)

    U,p = mannwhitneyu(group0,group1,alternative='two-sided')
    score = U/(n0*n1) #P(valeur churn=0 > valeur churn=1)
    res.append({
        'variable': col,
        'test': 'Mann-Whitney',
        'statistique': U,
        'p_value': p,
        'score' : score,
    })

    num_results = pd.DataFrame(res)


In [ ]:
num_results

2. Test de Chi-2 pour les variables categoriques

> **$H_0$:** la variable et churn sont independants (la proportion de churn est la meme dans toutes les categories de la variable).

> **$H_1$:** la variable et churn sont associes (la proportion de churn differe selon la categorie).

In [ ]:
def cramersV(contingency_table):
    chi2,p,dof,expected = chi2_contingency(contingency_table)
    n = contingency_table.values.sum()
    min_dim = min(contingency_table.shape) - 1
    v = np.sqrt(chi2/(n*min_dim))
    return chi2,p,v
cat_res = []
for col in cat:
    table = pd.crosstab(data[col],data[churn])
    chi2,p,v = cramersV(table)
    cat_res.append({
        'variable': col,
        'test': 'Chi-2',
        'statistique': chi2,
        'p_value': p,
        'score' : v,
    })

    cat_results = pd.DataFrame(cat_res)


In [ ]:
cat_results

3. Correction de Benjamini-Hochberg

In [ ]:
# Ajustement des p_values
results = pd.concat([num_results, cat_results],ignore_index=True)
results['p_value_adj'] = false_discovery_control(results['p_value'],method='bh')
results.sort_values('p_value_adj')

- Interpretation

Apres correction, `genre` (p_adj=0.102) et `cohorte` (p_adj=0.531) restent non significatifs.
Toutes les autres variables restent significatives apres correction.
`age` et `nb_inscriptions_precedentes` ont un p_value_adj tres faible, mais leur score est proche de 0.5 (0.533 et 0.485), ce qui signifie une capacite de discrimination quasi nulle entre churn=0 et churn=1. A n=12 888, un ecart infime suffit a produire un p-value tres faible sans que cela reflete un effet reel.
Pour les variables categoriques, comme churn n'a que 2 modalites, le V de Cramer suit directement les seuils de Cohen (negligeable <0.1, faible 0.1-0.3, moyen 0.3-0.5). Meme la variable la plus discriminante, `niveau_scolaire` (V=0.176), reste dans la zone "faible". `region`, `situation_handicap` et `annee_scolaire` sont sous le seuil de 0.1.


In [ ]:
insignificant = ['genre','cohorte']
data.drop(insignificant,axis='columns', inplace=True)
for col in insignificant:
    cat.remove(col)
print(cat)
data.head()

## Correlation

1. Pour les variables numeriques

In [ ]:
corr = data[num].corr(method='spearman')
corr

In [ ]:
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=0, vmax=1)
plt.show()

- `total_volume` et `distinct_active_days` sont trop correlees entre eux, on va donc travailler avec `total_volume` seule.

In [ ]:
data.drop(columns=['distinct_active_days'], inplace=True)
num.remove('distinct_active_days')

In [ ]:
active = data[data['total_volume'] > 0]
sns.heatmap(active[num].corr(method='spearman'), annot=True, cmap='coolwarm', vmin=0, vmax=1)
plt.show()

2. Pour les variables categoriques

In [ ]:
cramers_matrix = pd.DataFrame(index=cat, columns=cat, dtype=float)
for col1 in cat:
    for col2 in cat:
        table = pd.crosstab(data[col1], data[col2])
        _, _, v = cramersV(table)
        cramers_matrix.loc[col1, col2] = v

cramers_matrix = cramers_matrix.astype(float)
sns.heatmap(cramers_matrix, annot=True, cmap='coolwarm', vmin=0, vmax=1)
plt.show()

### Conclusion

- Le test de Mann-Whitney confirme un ecart significatif entre churn=0 et churn=1 pour les nouvelles variables comportementales et academiques : `total_volume` (score=0.618), `distinct_active_days` (score=0.617), `n_evaluations` (score=0.581) et `days_since_last_activity` (score=0.374). Ces variables discriminent reellement les deux groupes, contrairement a `age` (score=0.533) et `nb_inscriptions_precedentes` (score=0.485), dont le score reste proche de 0.5 malgre un p_value tres faible, comme deja constate dans l'analyse initiale.

- `total_volume` et `distinct_active_days` sont quasi redondantes (rho=0.982) : seule `total_volume` est conservee.

- `total_volume` et `days_since_last_activity` sont fortement correlees sur l'ensemble des inscriptions (rho=-0.908), mais ce chiffre chute a -0.464 une fois restreint aux inscriptions avec activite (`total_volume > 0`). Une bonne partie de la correlation globale provient donc du bloc partage des inscriptions sans aucune activite (`total_volume=0`, `distinct_active_days=0` et `days_since_last_activity=60` simultanement, par construction). L'association residuelle (-0.464) reste reelle mais moderee : les deux variables sont conservees, car elles capturent des aspects distincts de l'engagement — volume cumule d'un cote, recence de l'autre.

- `n_evaluations` est moderement correle a `total_volume` (rho=0.672 global, 0.504 restreint) et a `days_since_last_activity` (rho=-0.541 global, -0.125 restreint). L'association est attendue mais pas assez forte pour justifier une exclusion.

- L'ensemble numerique retenu pour la modelisation est : `total_volume`, `days_since_last_activity`, `n_evaluations`, `age`, `nb_inscriptions_precedentes`. Une verification par VIF sera necessaire une fois les variables categoriques encodees, avant l'entrainement des modeles.

- `avg_note` et `avg_lateness_days` ont ete exclues plus tot dans le notebook (~73% de valeurs manquantes, liees structurellement au fait que la premiere echeance mediane arrive apres T_obs=60 jours). `n_evaluations` porte le signal d'engagement academique dans la fenetre, sans ce cout de donnees manquantes.

- Parmi les variables categoriques, `niveau_scolaire` et `annee_scolaire` presentent l'association la plus elevee (V=0.378), un niveau modere. Les autres paires restent sous 0.1, donc negligeables.
